# Policy Gradient Methods

Up to now, every method we have studied is **value-based**. The agent learns a value function, and the policy is then derived from it by acting greedily, or epsilon-greedily, with respect to the estimated values. The policy has no independent existence: without the value function, there is no behaviour to execute.

In this chapter we take the opposite point of view. We **parameterise the policy directly** and adjust its parameters so that the agent collects more reward. The value function becomes optional. It can help, as we will see at the end of the chapter, but the object we learn is the policy itself.

A parameterised policy assigns a probability to each action in each state. The parameter vector is denoted by

$\displaystyle \theta$

and the policy is written as

$\displaystyle \pi_\theta(a \mid s) = P(A_t = a \mid S_t = s;\ \theta)$

Learning means choosing the parameters so as to maximise a **performance measure**, written as

$\displaystyle J(\theta)$

The natural update is **gradient ascent**:

$\displaystyle \theta \leftarrow \theta + \alpha\, \nabla_\theta J(\theta)$

The only structural requirement is differentiability: the action probability must change smoothly when the parameters change. In symbols, the differentiable quantity is

$\displaystyle \pi_\theta(a \mid s)$

In practice we also want the policy to remain **stochastic** during learning, so that the agent keeps trying different actions. For a discrete action space, a convenient parameterisation is the **softmax** over action preferences:

$\displaystyle \pi_\theta(a \mid s) = \frac{e^{h(s,a,\theta)}}{\sum_{b} e^{h(s,b,\theta)}}$

The function

$\displaystyle h(s,a,\theta)$

is the **preference**, or logit, of action

$\displaystyle a$

in state

$\displaystyle s$

computed by a neural network with weights

$\displaystyle \theta$

We already met the softmax in Chapter 04, as an exploration strategy that turned *value estimates* into probabilities. Here the preferences are **not** value estimates. They are free parameters, and the network learns them so as to make good actions more likely, without ever having to say *how much* an action is worth.

The chapter proceeds as follows. We first ask why one would want to learn the policy directly. We then derive the **policy gradient theorem**, which tells us how to estimate

$\displaystyle \nabla_\theta J(\theta)$

from sampled trajectories, and we implement the resulting algorithm, **REINFORCE**, on Cart-Pole. We then study its main weakness, the variance of the gradient estimate, and reduce it with a learned **baseline**, obtaining the **Vanilla Policy Gradient** algorithm, which we test on Cart-Pole and on Lunar Lander.


## Why Learn the Policy Directly?

Value-based methods have served us well, so the first question is why a different approach is needed at all. There are four main reasons.

**Stochastic policies can be optimal.** A value-based agent always ends up with a (near-)deterministic policy, because it selects the action with the highest value. In a fully observable Markov decision process this is enough: there always exists a deterministic optimal policy. But when the agent **cannot fully observe the state**, two different situations may produce the same observation, and the best thing to do may be to act *randomly*, with carefully chosen probabilities. Consider a foggy version of the Frozen Lake environment, in which some cells look identical:

![](./images/stochastic-policies.png)

A deterministic policy that always turns left in the foggy cells fails in half of them; a policy that goes left or right with probability one half reaches the goal from all of them. A parameterised stochastic policy can represent this behaviour directly, while a greedy policy over values cannot. Note that in this case the policy is not "hedging" against the environment's randomness, but against **the agent's own uncertainty about where it is**.

**The policy can be simpler than the value function.** Value functions carry more information than is needed to act. Consider a long deterministic corridor with a goal at both ends:

![](./images/too-much-informative.png)

The optimal policy is trivial: move towards the nearest goal, and choose randomly in the middle. The optimal value function is more complicated, because it assigns a different number to every cell. A value-based agent must estimate all those numbers accurately enough to rank the two actions correctly. Learning the mapping "state to action preference" is a much easier function approximation problem than learning "state to exact expected return".

**Smooth policy changes.** In a value-based method, a small change in the value estimates can flip the greedy action abruptly. The operation responsible for this discontinuity is

$\displaystyle \arg\max_a Q(s,a)$

A small numerical change can therefore produce a different action, a different trajectory, different data, and then different value estimates. This feedback can oscillate or even diverge, as we discussed with the deadly triad in Chapter 07. A policy gradient method changes the action probabilities **gradually**, by small steps along the gradient. This smoothness is what gives policy gradient methods their convergence guarantees: under suitable step sizes, they converge at least to a local optimum of the performance measure.

**Large and continuous action spaces.** Every value-based method we have seen needs to solve

$\displaystyle \max_a Q(s,a)$

This is immediate with two actions, but it becomes hard or impossible when actions are continuous, such as the torque applied to a robot joint. A parameterised policy can output the action, or the parameters of a distribution over actions, directly. We will exploit this in Chapter 10.

There is of course a price to pay. Policy gradient methods are **on-policy**: the gradient must be estimated with data generated by the current policy, so experience cannot be stored in a replay memory and reused as in DQN. Their gradient estimates have **high variance**, and they are only guaranteed to find a **local** optimum. Much of this chapter and of the next ones is about mitigating these problems.


## The Objective Function

To do gradient ascent we first need to define precisely what we want to maximise. In an episodic task, the most natural performance measure is the **expected discounted return from the start of an episode**:

$\displaystyle J(\theta) = \mathbb{E}_{S_0 \sim p_0}\big[ v_{\pi_\theta}(S_0) \big]$

Here

$\displaystyle p_0$

is the distribution of the initial state. It is convenient to rewrite the same objective in terms of whole trajectories. We denote a trajectory by

$\displaystyle \tau = (S_0, A_0, R_1, S_1, A_1, R_2, \ldots, S_{T-1}, A_{T-1}, R_T, S_T)$

and its discounted return by

$\displaystyle G(\tau) = \sum_{t=0}^{T-1} \gamma^{t} R_{t+1}$

The performance measure is therefore the expected return over all trajectories that the current policy can generate:

$\displaystyle J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\big[ G(\tau) \big] = \sum_{\tau} p(\tau \mid \pi_\theta)\, G(\tau)$

The **probability of a trajectory** is the product of three kinds of terms: the probability of the initial state, the action probabilities chosen by the policy, and the transition probabilities chosen by the environment:

$\displaystyle p(\tau \mid \pi_\theta) = p_0(S_0) \prod_{t=0}^{T-1} \pi_\theta(A_t \mid S_t)\, p(S_{t+1} \mid S_t, A_t)$

This factorisation is the key to everything that follows. The policy appears only through factors of the form

$\displaystyle \pi_\theta(A_t \mid S_t)$

The environment appears through the initial-state distribution and the transition function,

$\displaystyle p_0, \qquad p(s' \mid s,a)$

and these do **not** depend on the policy parameters.


## The Policy Gradient Theorem

We need the gradient of the performance measure:

$\displaystyle \nabla_\theta J(\theta) = \nabla_\theta \sum_{\tau} p(\tau \mid \pi_\theta)\, G(\tau) = \sum_{\tau} \nabla_\theta p(\tau \mid \pi_\theta)\, G(\tau)$

This expression cannot be computed as it stands, for two reasons. First, the sum runs over all possible trajectories, which are far too many. Second, the trajectory probability contains the transition probabilities of the environment, which the agent does not know. We remove both obstacles with two manipulations.

**The log-derivative trick.** The useful identity is

$\displaystyle \nabla_\theta \log f(\theta) = \frac{\nabla_\theta f(\theta)}{f(\theta)}$

Equivalently,

$\displaystyle \nabla_\theta f(\theta) = f(\theta)\, \nabla_\theta \log f(\theta)$

Applying this identity to the trajectory probability gives

$\displaystyle \nabla_\theta J(\theta) = \sum_{\tau} p(\tau \mid \pi_\theta)\, \nabla_\theta \log p(\tau \mid \pi_\theta)\, G(\tau)$

or, as an expectation,

$\displaystyle \nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\big[ \nabla_\theta \log p(\tau \mid \pi_\theta)\, G(\tau) \big]$

The gradient is now written as an **expectation under the current policy**, so it can be estimated by sampling trajectories. This removes the first obstacle.

**The dynamics drop out.** Taking the logarithm of the trajectory probability turns the product into a sum:

$\displaystyle \log p(\tau \mid \pi_\theta) = \log p_0(S_0) + \sum_{t=0}^{T-1} \log \pi_\theta(A_t \mid S_t) + \sum_{t=0}^{T-1} \log p(S_{t+1} \mid S_t, A_t)$

Only the policy terms depend on the parameter vector. The initial-state distribution and the transition function are properties of the environment, so their gradients vanish:

$\displaystyle \nabla_\theta \log p(\tau \mid \pi_\theta) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(A_t \mid S_t)$

This removes the second obstacle: the unknown dynamics have disappeared, and only the policy, which we control, remains. Substituting, we obtain the **policy gradient theorem** in trajectory form:

$\displaystyle \nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(A_t \mid S_t)\, G(\tau) \right]$

The term

$\displaystyle \nabla_\theta \log \pi_\theta(a \mid s)$

is called the **score function**. It is the direction in parameter space that most rapidly increases the log-probability of action

$\displaystyle a$

in state

$\displaystyle s$

The theorem says that the gradient of the performance is the expected score of the actions taken, **weighted by the return of the trajectory**. The intuition is simple: trajectories with high return push the parameters so that the actions taken along them become more likely; trajectories with low or negative return make them less likely. The agent never needs to know *why* a trajectory was good, nor how the environment works.


For a softmax policy the score has a particularly transparent form. Suppose the preferences are linear in a feature vector:

$\displaystyle h(s,a,\theta) = \theta^\top x(s,a)$

Then the score is

$\displaystyle \nabla_\theta \log \pi_\theta(a \mid s) = x(s,a) - \sum_{b} \pi_\theta(b \mid s)\, x(s,b)$

That is, the update compares the feature vector of the action taken with the feature vector expected under the current policy. Chapter 07 wrote feature vectors as

$\displaystyle \phi(s,a)$

From this chapter on, we reserve

$\displaystyle \phi$

for the parameters of the value network, and write action features as

$\displaystyle x(s,a)$

Increasing the probability of one action necessarily decreases that of the others. With a neural network, the same quantity is computed by backpropagation.

Finally, since the gradient is an expectation, we estimate it with the **sample mean** over a batch of trajectories collected by running the current policy. If the batch contains

$\displaystyle m$

trajectories, the estimate is

$\displaystyle \nabla_\theta J(\theta) \approx \frac{1}{m} \sum_{i=1}^{m} \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta\big(a_t^{(i)} \mid s_t^{(i)}\big)\, G\big(\tau^{(i)}\big)$

This estimator is **unbiased**: on average it points in the direction of the true gradient. Its weakness, as we will see, is its **variance**.

*Remark.* In the literature, the policy gradient theorem is often stated in a different but equivalent form, in terms of the distribution of states visited under the policy:

$\displaystyle \mu_\pi(s)$

The theorem is then written as

$\displaystyle \nabla_\theta J(\theta) \propto \sum_{s} \mu_\pi(s) \sum_{a} q_\pi(s,a)\, \nabla_\theta \pi_\theta(a \mid s)$

or, using the score-function form,

$\displaystyle \nabla_\theta J(\theta) \propto \mathbb{E}_{\pi}\big[ q_\pi(S_t, A_t)\, \nabla_\theta \log \pi_\theta(A_t \mid S_t) \big]$

The trajectory form derived above is the one we implement, because sampled returns are the natural way to estimate the action values that appear in this expression.


### Causality

In the formula above, the return of the **whole** trajectory multiplies the score of **every** action. This means that an action taken near the end of an episode is weighted by rewards collected near the beginning. That cannot be causal: an action taken at time

$\displaystyle t$

cannot influence a reward received at time

$\displaystyle k \le t$

We can make this intuition precise. For any fixed state, the expected score of the action sampled from the policy is zero:

$\displaystyle \mathbb{E}_{A_t \sim \pi_\theta}\big[ \nabla_\theta \log \pi_\theta(A_t \mid S_t) \,\big|\, S_t \big] = \sum_a \pi_\theta(a \mid S_t)\, \frac{\nabla_\theta \pi_\theta(a \mid S_t)}{\pi_\theta(a \mid S_t)} = \nabla_\theta \sum_a \pi_\theta(a \mid S_t) = \nabla_\theta 1 = 0$

Therefore any past reward has zero expected contribution to the gradient term for the current action:

$\displaystyle \mathbb{E}\big[ \nabla_\theta \log \pi_\theta(A_t \mid S_t)\, R_k \big] = 0 \qquad \text{for } k \le t$

Past rewards add noise but no signal. We can therefore drop them and weight each action only with the rewards that **follow** it:

![](./images/causal.png)


The quantity that remains is the familiar **return from time step**

$\displaystyle t$

also called the *reward-to-go*:

$\displaystyle G_t = \sum_{k=t}^{T-1} \gamma^{\,k-t} R_{k+1} = R_{t+1} + \gamma\, G_{t+1}$

The practical estimator used by REINFORCE becomes

$\displaystyle \nabla_\theta J(\theta) \approx \frac{1}{m} \sum_{i=1}^{m} \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta\big(a_t^{(i)} \mid s_t^{(i)}\big)\, G_t^{(i)}$

The estimator has **lower variance**, because each term now contains fewer random rewards. There is one technical detail to keep in mind. For the discounted start-state objective defined above, the exact causal gradient includes an additional factor:

$\displaystyle \gamma^t$

Many implementations, including the one below, omit this factor. This is standard practice: otherwise late actions in long episodes would receive extremely small updates. The resulting update is best understood as the usual practical policy-gradient estimator for episodic tasks.

The recursive form of the return,

$\displaystyle G_t = R_{t+1} + \gamma G_{t+1}$

is the same structure that underlies the Bellman equations, and it lets us compute all returns of an episode in a single backward pass.


## REINFORCE

The algorithm that implements the estimator above is [**REINFORCE**](./papers/1992%20-%20Simple%20Statistical%20Gradient-Following%20Algorithms%20for%20Connectionist%20Reinforcement%20Learning.pdf), introduced by Ronald Williams in 1992, well before deep learning. It is a **Monte Carlo** method in the sense of Chapter 05: it waits until the end of an episode, computes the actual returns,

$\displaystyle G_t$

and uses them to update the policy. As in Chapter 07, we use the Cart-Pole environment as a first test bed, so that we can compare the results with those of the value-based methods:


In [ ]:
import gymnasium

cart_pole = gymnasium.make('CartPole-v1', render_mode="rgb_array")

print('State Space: ', cart_pole.observation_space.shape)
print('Action Space: ', cart_pole.action_space.n)

We fix the random seeds for reproducibility:

In [ ]:
import torch
import numpy as np
import random

# set the seeds for reproducibility of results
seed = 34

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

cart_pole.reset(seed=seed)
cart_pole.action_space.seed(seed);

### Policy Network

The parametric policy is a neural network that takes the state as input and outputs one **preference** for each action. We use the same architecture as in Chapter 07 (two hidden layers with ReLU activations), but the meaning of the output is different: these are not Q-values, they are the logits of a softmax. The network reads the sizes of the state and action spaces from the environment, so that we can reuse it on other environments later in the chapter:

In [ ]:
def create_policy_network(env, first_hidden_layer=128, second_hidden_layer=64):

    # size of the state vector and number of actions
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n

    # a fully connected network with two hidden layers
    dnn = torch.nn.Sequential(
        torch.nn.Linear(state_size, first_hidden_layer),
        torch.nn.ReLU(),
        torch.nn.Linear(first_hidden_layer, second_hidden_layer),
        torch.nn.ReLU(),
        # output layer: one preference (logit) per action
        torch.nn.Linear(second_hidden_layer, action_size),
    )

    return dnn

pi = create_policy_network(cart_pole)

The softmax that turns preferences into probabilities does not need to be part of the network: PyTorch provides a `Categorical` distribution that accepts the logits directly and offers everything we need: sampling an action, the log-probability of the sampled action, and (later) the entropy of the distribution:

In [ ]:
def policy_distribution(state):

    # convert the state into a tensor
    state = torch.as_tensor(state, dtype=torch.float32)

    # compute the action preferences with the policy network
    preferences = pi(state)

    # the softmax over the preferences defines the action distribution
    return torch.distributions.Categorical(logits=preferences)

In policy-based methods there is **no need for a separate exploration strategy**: the policy itself is a probability distribution, and sampling an action from it is already exploratory. During training we sample; the function also returns the log-probability of the sampled action, which is the quantity whose gradient appears in the policy gradient theorem:

In [ ]:
def select_action(state):

    # the action distribution in the current state
    dist = policy_distribution(state)

    # sample an action (this is where exploration happens)
    action = dist.sample()

    # log-probability of the sampled action, needed for the gradient
    logpa = dist.log_prob(action)

    return int(action.item()), logpa

There are two ways to **evaluate** the learned policy. The first is to evaluate the stochastic policy itself, sampling the actions exactly as during training. This is the policy whose performance we are optimising:

$\displaystyle J(\theta)$

The second is to act greedily, selecting the most probable action in every state, as we did for the policy derived from Q-values in Chapter 07. The greedy policy is a different policy, which we never optimised directly. It is usually at least as good as the stochastic one, because it removes the randomness used for exploration, but, as we will see on Lunar Lander, this is not guaranteed. We define both, as functions of the state, so that they can be passed to the evaluation function; they will serve for all the algorithms of this chapter, since they all share the policy network:


In [ ]:
def stochastic_pi(state):

    # sample an action from the policy, as during training
    return select_action(state)[0]


def greedy_pi(state):

    # compute preferences without tracking gradients
    with torch.no_grad():
        preferences = pi(torch.as_tensor(state, dtype=torch.float32))

    # select the most probable action
    return int(torch.argmax(preferences).item())

We reuse the evaluation and visualisation functions of Chapter 07. The first runs a policy for a number of episodes and returns the average total reward; the second shows the policy in action:

In [ ]:
def evaluate(env, policy, episodes=1):

    # collect total rewards per episode
    rewards = []

    for episode in range(episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        total_reward = 0.0

        # run one episode
        while not done:
            action = policy(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

        rewards.append(total_reward)

    # return the average reward over episodes
    return float(np.mean(rewards))

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

def show_policy(env, policy, n_episodes=1, max_steps=500):

    for episode in range(n_episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        step = 0

        # run the episode until done or max_steps
        while not done and step < max_steps:

            # make a step
            action = policy(state)
            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # render the environment
            clear_output(wait=True)
            plt.imshow(env.render())
            plt.axis("off")
            plt.title(f"Episode {episode+1}, Step {step}")
            plt.show()
            plt.close()

            step += 1

As a reference, the random policy of Chapter 07 obtains an average score of about 22 on Cart-Pole. Before training, the policy network, randomly initialised, behaves essentially in the same way:

In [ ]:
random_score = evaluate(cart_pole, lambda state: cart_pole.action_space.sample(), episodes=100)
initial_score = evaluate(cart_pole, greedy_pi, episodes=100)

print("Random score:  ", random_score)
print("Initial score: ", initial_score)

### Loss Function

Deep learning libraries are built to **minimise** a loss by gradient descent, while we want to **maximise** the performance measure by gradient ascent:

$\displaystyle J(\theta)$

The two are reconciled by defining the loss as the negative of the quantity whose gradient is the policy-gradient estimate. For a single trajectory:

$\displaystyle \mathcal{L}(\theta) = - \frac{1}{T} \sum_{t=0}^{T-1} G_t\, \log \pi_\theta(A_t \mid S_t)$

When differentiating this expression, the returns are treated as constants. They are numbers observed from the environment, not functions of the policy parameters:

$\displaystyle G_t \not\equiv G_t(\theta)$

The gradient of the loss is therefore the negative of the policy-gradient estimator, and a gradient **descent** step on the loss is a gradient **ascent** step on the performance measure. The loss is **not a measure of how good the policy is**. Unlike the mean squared TD error of the previous chapter, its value tells us little about learning progress; it is only a device that produces the right gradient.

The division by the episode length keeps the size of the update comparable between short and long episodes. It is a constant factor for the episode and could be absorbed into the learning rate.

The returns are computed in a single backward pass over the rewards of the episode, using the recursion

$\displaystyle G_t = R_{t+1} + \gamma G_{t+1}$

with terminal condition

$\displaystyle G_T = 0$


In [ ]:
def compute_returns(rewards, gamma):

    # G_t = R_{t+1} + gamma * G_{t+1}, computed backwards from the end of the episode
    returns = np.zeros(len(rewards), dtype=np.float32)
    future_return = 0.0

    for t in reversed(range(len(rewards))):
        future_return = rewards[t] + gamma * future_return
        returns[t] = future_return

    return returns

We need an optimizer for the policy network. As in Chapter 07, we use RMSprop:

In [ ]:
learning_rate = 0.0005
optimizer = torch.optim.RMSprop(pi.parameters(), lr=learning_rate)

The optimization step receives the rewards and the log-probabilities collected during one episode, computes the returns, builds the loss, and performs one gradient step:

In [ ]:
def optimize(rewards, logpas, gamma=0.99):

    # returns from each time step
    returns = torch.as_tensor(compute_returns(rewards, gamma))

    # log-probabilities of the actions taken, as a single tensor
    logpas = torch.stack(logpas)

    # the policy gradient loss: minus the return-weighted log-probabilities
    loss = -(returns * logpas).mean()

    # one step of gradient descent on the loss (= gradient ascent on J)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

### Algorithm

We can now put the pieces together. REINFORCE alternates between collecting one complete episode with the current policy and performing one gradient step:

1. Use a neural network that takes the state as input and outputs one preference per action; the softmax of the preferences is the policy
2. Run one full episode, sampling actions from the policy, and record the rewards and the log-probabilities of the actions taken
3. At the end of the episode, compute the return from every time step:

$\displaystyle G_t$

4. Update the parameters with one step of gradient ascent on the return-weighted log-probabilities:

$\displaystyle \sum_t G_t \log \pi_\theta(A_t \mid S_t)$

5. Discard the episode and repeat with the updated policy

The episode ends both when the pole falls, called *termination*, and when the time limit is reached. For Cart-Pole this limit is

$\displaystyle 500$

steps. A Monte Carlo method has no value estimate to fall back on, so it treats the two cases in the same way, as if the return after the time limit were zero. This is a small approximation on Cart-Pole, and the next chapter will show how a critic handles the two cases differently.

The last point deserves attention. The data collected with the old policy cannot be reused after the update: the gradient estimate is only valid for the policy that generated the trajectories. This is what makes REINFORCE an **on-policy** algorithm, and it is the main reason for its sample inefficiency.


In [ ]:
def reinforce(env, max_episodes, gamma=0.99):

    scores = []

    for episode in range(max_episodes):

        # reset the environment
        state, _ = env.reset()
        done = False

        # the rewards and log-probabilities collected along the episode
        rewards = []
        logpas = []

        # run one full episode with the current policy
        while not done:
            action, logpa = select_action(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            rewards.append(reward)
            logpas.append(logpa)

        # one optimization step at the end of the episode
        optimize(rewards, logpas, gamma)

        # evaluate the greedy policy after each episode
        score = evaluate(env, greedy_pi, episodes=10)
        scores.append(score)

        print(f"Episode {episode + 1:04}, score {score:05.1f}", end="\r", flush=True)

    return scores

### Solve Cart-Pole

We can train the agent on the Cart-Pole environment:

In [ ]:
scores = reinforce(cart_pole, max_episodes=1000)

We evaluate the trained agent against the random baseline:

In [ ]:
agent_score = evaluate(cart_pole, greedy_pi, episodes=10)

print("Random score", random_score)
print("Agent score", agent_score)

And we can see the agent in action:

In [ ]:
show_policy(cart_pole, greedy_pi, n_episodes=1, max_steps=500)

As in Chapter 07, a single run tells us little about **how** the algorithm learns and **how reliable** it is. We reuse the experiment function of Chapter 07, which trains the agent from scratch with several random seeds, smooths the learning curves, and saves the results in the `temp` folder, so that they can be reloaded without repeating the training:

In [ ]:
import joblib
from pathlib import Path

def experiment(
    env,
    train,
    setup,
    max_episodes,
    name,
    seeds=(12, 34, 56, 78, 90),
    smoothing_window=25,
):

    results_path = Path("temp") / f"{name}_results.joblib"

    # load saved results if they already exist
    if results_path.exists():
        experiment_results = joblib.load(results_path)

        n_points = len(experiment_results["mean_score"])
        final_mean = experiment_results["mean_score"][-1]
        final_min = experiment_results["min_score"][-1]
        final_max = experiment_results["max_score"][-1]

        print(f"Loaded saved results from {results_path}")
        print(f"Smoothed points: {n_points}")
        print(f"Smoothing window: {experiment_results['smoothing_window']}")
        print(f"Final score: mean={final_mean:.1f}, range=[{final_min:.1f}, {final_max:.1f}]")

        return experiment_results

    results = []

    for seed in seeds:
        print(f"Experiment seed: {seed}")

        # set seeds for reproducibility
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        env.reset(seed=seed)
        env.action_space.seed(seed)

        # initialize networks and optimizers
        setup()

        # train the agent
        scores = train(env, max_episodes=max_episodes)

        # smooth the learning curve
        kernel = np.ones(smoothing_window) / smoothing_window
        scores = np.convolve(scores, kernel, mode="valid")

        results.append(scores)
        print()

    results = np.array(results)

    experiment_results = {
        "max_score": np.max(results, axis=0),
        "min_score": np.min(results, axis=0),
        "mean_score": np.mean(results, axis=0),
        "smoothing_window": smoothing_window,
        "seeds": seeds,
        "max_episodes": max_episodes,
    }

    results_path.parent.mkdir(exist_ok=True)
    joblib.dump(experiment_results, results_path)

    return experiment_results

We define the hyperparameters and the setup function that creates a fresh policy network and optimizer for each seed, and we run the experiment:

In [ ]:
# define the experiment setup and hyperparameters
gamma = 0.99               # discount factor
learning_rate = 0.0005     # optimizer step size
first_hidden_layer = 128   # size of the first hidden layer
second_hidden_layer = 64   # size of the second hidden layer

def setup_reinforce():
    global pi, optimizer
    pi = create_policy_network(cart_pole, first_hidden_layer, second_hidden_layer)
    optimizer = torch.optim.RMSprop(pi.parameters(), lr=learning_rate)

# run the experiment
reinforce_results = experiment(
    env=cart_pole,
    train=reinforce,
    setup=setup_reinforce,
    max_episodes=1500,
    name="reinforce",
)

We plot the mean learning curve across seeds, together with the range between the best and the worst run, with the same functions used in Chapter 07:

In [ ]:
def plot_scores(results, title="Score", color="orange"):

    # the smoothed curves start after the first smoothing window
    offset = results.get("smoothing_window", 1)
    episodes = np.arange(len(results["mean_score"])) + offset

    plt.figure(figsize=(12, 6))
    plt.title(title)
    plt.xlabel("Episodes")
    plt.ylabel("Score")

    plt.plot(episodes, results["mean_score"], color=color, linewidth=2, label="Mean over seeds")
    plt.fill_between(episodes, results["min_score"], results["max_score"], color=color, alpha=0.3, label="Range over seeds")

    plt.legend(loc="upper left")
    plt.show()

plot_scores(reinforce_results, title="REINFORCE score")

In [ ]:
def plot_comparison(
    first_results,
    second_results,
    first_label="A",
    second_label="B",
    first_color="blue",
    second_color="red",
):

    first_episodes = np.arange(len(first_results["mean_score"])) + first_results.get("smoothing_window", 1)
    second_episodes = np.arange(len(second_results["mean_score"])) + second_results.get("smoothing_window", 1)

    plt.figure(figsize=(12, 6))
    plt.title(f"{second_label} vs {first_label}")
    plt.xlabel("Episodes")
    plt.ylabel("Score")

    # plot the first algorithm
    plt.plot(first_episodes, first_results["mean_score"], color=first_color, linewidth=2, label=f"{first_label} mean")
    plt.fill_between(first_episodes, first_results["min_score"], first_results["max_score"], color=first_color, alpha=0.20, label=f"{first_label} range")

    # plot the second algorithm
    plt.plot(second_episodes, second_results["mean_score"], color=second_color, linewidth=2, label=f"{second_label} mean")
    plt.fill_between(second_episodes, second_results["min_score"], second_results["max_score"], color=second_color, alpha=0.20, label=f"{second_label} range")

    plt.legend()
    plt.show()

It is instructive to compare REINFORCE with the value-based agents of Chapter 07 on the same environment. If the results of the DQN experiment are available in the `temp` folder of Chapter 07, we can plot them together:

In [ ]:
dqn_results_path = Path("../07 - Generalizing with Neural Networks/temp/dqn_results.joblib")

if dqn_results_path.exists():
    dqn_results = joblib.load(dqn_results_path)
    plot_comparison(dqn_results, reinforce_results, first_label="DQN", second_label="REINFORCE", first_color="blue", second_color="orange")
else:
    print("DQN results not found: run the experiments of Chapter 07 first.")

Two features of the REINFORCE curve are worth noting. First, learning is **slow**: compared with DQN, which improves within a few dozen episodes, REINFORCE needs hundreds of episodes before the score rises substantially, because every episode produces exactly one gradient step and is then thrown away. Second, learning is **noisy**: the mean curve fluctuates and the range across seeds is wide, and it is not unusual for a run to lose a good policy after having found it.

Both features have the same origin: the gradient estimate is **unbiased but has high variance**. Every random factor along a trajectory, from the initial state to the sampled actions and stochastic transitions, ends up in the returns:

$\displaystyle G_t$

The return is therefore a very noisy training signal. On the positive side, REINFORCE learns with no replay memory, no target network, and no exploration schedule: a single network and a loss function of two lines. It is too inefficient to be practical on complex problems, but it remains the conceptual reference for every method in the rest of the course.


## Reducing Variance with a Baseline

The core idea of the policy gradient is to increase the probability of actions that lead to high returns and decrease that of actions that lead to low returns. But "high" and "low" compared to what? In Cart-Pole all rewards are positive, so every return is positive and **every action taken is reinforced**. A good action is reinforced a bit more than a bad one, and the policy improves only because of this difference. The common positive part of the returns is pure noise that inflates the variance of the estimate.

The remedy is to subtract from the return a reference value, called a **baseline**. The baseline may depend on the state, but not on the action:

$\displaystyle b(S_t)$

The gradient estimator becomes

$\displaystyle \nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(A_t \mid S_t)\, \big( G_t - b(S_t) \big) \right]$

This does **not** change the expected value of the estimator. The subtracted term has zero mean, by the same score-function argument we used for causality:

$\displaystyle \mathbb{E}_{A_t \sim \pi_\theta}\big[ \nabla_\theta \log \pi_\theta(A_t \mid S_t)\, b(S_t) \,\big|\, S_t \big] = b(S_t) \sum_a \nabla_\theta \pi_\theta(a \mid S_t) = b(S_t)\, \nabla_\theta 1 = 0$

So any state-dependent baseline leaves the estimator **unbiased**, while a well-chosen one can reduce its variance dramatically.

Which baseline should we choose? The natural candidate is the **state-value function**, the return we expect from the current state under the current policy:

$\displaystyle v_\pi(S_t)$

Recall from Chapter 03 that the Monte Carlo return is an unbiased sample of the action value:

$\displaystyle \mathbb{E}_\pi[G_t \mid S_t, A_t] = q_\pi(S_t, A_t)$

Therefore subtracting the state value produces a sample of the **advantage function**, which we met in Chapter 07 with the dueling architecture:

$\displaystyle a_\pi(s,a) = q_\pi(s,a) - v_\pi(s)$

The advantage measures how much better an action is than the average action in that state. It is **centred around zero**: better-than-average actions have positive advantage and are reinforced, worse-than-average actions have negative advantage and are discouraged. This is exactly the comparison that was missing.

Of course, we do not know the true state-value function. We will **learn an estimate** with a second neural network, with its own parameters:

$\displaystyle V(s;\phi)$

and use the sample advantage

$\displaystyle A(S_t, A_t) = G_t - V(S_t;\phi)$

to weight the score of each action.


## Vanilla Policy Gradient (VPG)

The algorithm obtained by adding a learned baseline to REINFORCE is usually called **Vanilla Policy Gradient** (or *REINFORCE with baseline* in Sutton & Barto's terminology). It requires two networks: the **policy network**, with one output per action, and the **value network**, with a single output:

![](./images/networks.png)

### Learning the Baseline

The value network must predict the return observed from each state. This is a plain **regression** problem: the target is the Monte Carlo return,

$\displaystyle G_t$

and the loss is the mean squared error over the steps of the episode:

$\displaystyle \mathcal{L}_v(\phi) = \frac{1}{T} \sum_{t=0}^{T-1} \big( G_t - V(S_t;\phi) \big)^2$

It is worth pausing on a difference with Chapter 07. There, the regression target was *bootstrapped*: it contained the network's own prediction for the next state, and we had to treat it as a constant, using the semi-gradient trick. Here the target is a **complete, observed return**. It is a genuine constant with respect to the value-network parameters:

$\displaystyle G_t \not\equiv G_t(\phi)$

The value network is therefore trained by ordinary supervised learning, and the baseline does not introduce any of the instabilities of bootstrapping. The price is that we must wait for the end of the episode, and that the targets are as noisy as the returns.

### Learning the Policy

The policy loss is the same as in REINFORCE, with the return replaced by the sample advantage:

$\displaystyle \mathcal{L}_\pi(\theta) = - \frac{1}{T} \sum_{t=0}^{T-1} \big( G_t - V(S_t;\phi) \big) \log \pi_\theta(A_t \mid S_t)$

When we differentiate this loss with respect to the policy parameters, the advantage must be treated as a constant. It is a weight on the score, not a function we want to optimise through. In PyTorch this is the role of `detach()`, which prevents the gradient of the policy loss from flowing into the value network.


### Entropy Regularisation

There is one more improvement that is almost always used with policy gradient methods. We want the policy to exploit what it has learned, but also to keep exploring, to avoid converging prematurely to a suboptimal action. In practice, however, the action distribution often **collapses** early onto a single action, especially when the first returns are noisy or misleading. Once one action has probability close to one,

$\displaystyle \pi_\theta(a \mid s) \approx 1$

the other actions are almost never sampled, their scores are almost never updated, and the policy may not recover. The agent has become **overconfident**.

A natural measure of how spread out a distribution is, is its **entropy**:

$\displaystyle H\big(\pi_\theta(\cdot \mid s)\big) = - \sum_{a} \pi_\theta(a \mid s)\, \log \pi_\theta(a \mid s)$

The entropy is maximal for the uniform distribution. With two actions, the maximum is

$\displaystyle \log |\mathcal{A}| = \log 2 \approx 0.69$

and it is zero for a deterministic policy. To discourage premature collapse, we add the entropy, weighted by a coefficient, to the objective we maximise:

$\displaystyle \beta$

The regularised objective has gradient estimated by

$\displaystyle \nabla_\theta \Big[ J(\theta) + \beta\, \mathbb{E}_{\pi_\theta}\big[ H(\pi_\theta(\cdot \mid S_t)) \big] \Big] \approx \frac{1}{T} \sum_{t=0}^{T-1} \nabla_\theta \Big[ \log \pi_\theta(A_t \mid S_t)\, A(S_t, A_t) + \beta\, H\big(\pi_\theta(\cdot \mid S_t)\big) \Big]$

Since the entropy is a differentiable function of the preferences, its gradient is computed by backpropagation like everything else. The coefficient controls the trade-off. If it is too large, the policy stays close to uniform and never exploits; if it is zero, we are back to the unregularised algorithm. Small values are typical, for example

$\displaystyle \beta = 0.01$

The entropy bonus is the policy-gradient counterpart of the exploration strategies of Chapter 04: it keeps the probabilities of the actions diverse for long enough to discover which ones are good.

*Is this an actor-critic method?* The value network is sometimes called a *critic*, and VPG is sometimes presented as the simplest actor-critic algorithm. We will reserve the name for the next chapter. The value network of VPG is used **only as a baseline**: it never appears in the target of the policy update, which is still the Monte Carlo return. The moment we use the value estimate to **replace** part of the return, that is to bootstrap, the algorithm changes nature, and this is precisely the step that defines the actor-critic family.


### Algorithm

The algorithm is the same as REINFORCE with three additions: a value network, an advantage estimate, and an entropy bonus.

1. Use two neural networks: a policy network with one preference per action, and a value network with a single output
2. Run one full episode, sampling actions from the policy; record rewards, log-probabilities, entropies, and the predicted values

$\displaystyle V(S_t;\phi)$

3. At the end of the episode, compute the returns and the sample advantages

$\displaystyle G_t, \qquad A(S_t, A_t) = G_t - V(S_t;\phi)$

4. Update the value network by regressing the prediction towards the Monte Carlo return

$\displaystyle V(S_t;\phi) \longrightarrow G_t$

5. Update the policy network with one step of gradient ascent on the advantage-weighted log-probabilities plus the entropy bonus

$\displaystyle \sum_t \Big[ A(S_t,A_t) \log \pi_\theta(A_t \mid S_t) + \beta H(\pi_\theta(\cdot \mid S_t)) \Big]$

The advantages are treated as constants in this update.

6. Discard the episode and repeat

The value network has the same architecture as the policy network, but a single output unit:


In [ ]:
def create_value_network(env, first_hidden_layer=128, second_hidden_layer=64):

    # size of the state vector
    state_size = env.observation_space.shape[0]

    # a fully connected network with two hidden layers
    dnn = torch.nn.Sequential(
        torch.nn.Linear(state_size, first_hidden_layer),
        torch.nn.ReLU(),
        torch.nn.Linear(first_hidden_layer, second_hidden_layer),
        torch.nn.ReLU(),
        # output layer: the estimated value of the state
        torch.nn.Linear(second_hidden_layer, 1),
    )

    return dnn

v = create_value_network(cart_pole)

Each network has its own optimizer:

In [ ]:
policy_optimizer = torch.optim.RMSprop(pi.parameters(), lr=learning_rate)
value_optimizer = torch.optim.RMSprop(v.parameters(), lr=learning_rate)

The action selection function now also returns the entropy of the action distribution in the current state:

In [ ]:
def select_action(state):

    # the action distribution in the current state
    dist = policy_distribution(state)

    # sample an action
    action = dist.sample()

    # log-probability of the sampled action and entropy of the distribution
    logpa = dist.log_prob(action)
    entropy = dist.entropy()

    return int(action.item()), logpa, entropy

The optimization step now updates both networks. The value loss is a mean squared error against the returns; the policy loss weights the log-probabilities by the *detached* advantages and adds the entropy bonus:

In [ ]:
def optimize(rewards, logpas, entropies, values, gamma=0.99, beta=0.01):

    # returns from each time step
    returns = torch.as_tensor(compute_returns(rewards, gamma))

    # stack the quantities collected along the episode
    logpas = torch.stack(logpas)
    entropies = torch.stack(entropies)
    values = torch.stack(values).squeeze(-1)

    # sample advantages: observed return minus predicted value
    advantages = returns - values

    # value loss: regression of V(s) towards the Monte Carlo return
    value_loss = advantages.pow(2).mean()

    # policy loss: advantages are treated as constants (detached)
    policy_loss = -(advantages.detach() * logpas + beta * entropies).mean()

    # update the value network
    value_optimizer.zero_grad()
    value_loss.backward()
    value_optimizer.step()

    # update the policy network
    policy_optimizer.zero_grad()
    policy_loss.backward()
    policy_optimizer.step()

    return policy_loss.item(), value_loss.item()

The training loop differs from REINFORCE only in what is recorded along the episode:

In [ ]:
def vpg(env, max_episodes, gamma=0.99, beta=0.01, eval_episodes=10):
    # eval_episodes > 0: evaluate the greedy policy after each episode;
    # eval_episodes = 0: score the stochastic policy with the return of the training episode

    scores = []

    for episode in range(max_episodes):

        # reset the environment
        state, _ = env.reset()
        done = False

        # the quantities collected along the episode
        rewards = []
        logpas = []
        entropies = []
        values = []

        # run one full episode with the current policy
        while not done:
            action, logpa, entropy = select_action(state)
            value = v(torch.as_tensor(state, dtype=torch.float32))

            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            rewards.append(reward)
            logpas.append(logpa)
            entropies.append(entropy)
            values.append(value)

        # one optimization step at the end of the episode
        optimize(rewards, logpas, entropies, values, gamma, beta)

        # evaluate the greedy policy after each episode, or use the return of the training episode
        if eval_episodes > 0:
            score = evaluate(env, greedy_pi, episodes=eval_episodes)
        else:
            score = float(np.sum(rewards))
        scores.append(score)

        print(f"Episode {episode + 1:04}, score {score:05.1f}", end="\r", flush=True)

    return scores

### Solve Cart-Pole

We train the agent on Cart-Pole:

In [ ]:
scores = vpg(cart_pole, max_episodes=1000)

In [ ]:
agent_score = evaluate(cart_pole, greedy_pi, episodes=10)

print("Random score", random_score)
print("Agent score", agent_score)

And we run the multi-seed experiment, creating fresh networks and optimizers for each seed:

In [ ]:
# define the experiment setup and hyperparameters
gamma = 0.99               # discount factor
beta = 0.01                # entropy coefficient
learning_rate = 0.0005     # optimizer step size
first_hidden_layer = 128   # size of the first hidden layer
second_hidden_layer = 64   # size of the second hidden layer

def setup_vpg():
    global pi, v, policy_optimizer, value_optimizer
    pi = create_policy_network(cart_pole, first_hidden_layer, second_hidden_layer)
    v = create_value_network(cart_pole, first_hidden_layer, second_hidden_layer)
    policy_optimizer = torch.optim.RMSprop(pi.parameters(), lr=learning_rate)
    value_optimizer = torch.optim.RMSprop(v.parameters(), lr=learning_rate)

# run the experiment
vpg_results = experiment(
    env=cart_pole,
    train=vpg,
    setup=setup_vpg,
    max_episodes=1500,
    name="vpg",
)

We compare the learning curves of REINFORCE and VPG:

In [ ]:
plot_comparison(
    reinforce_results,
    vpg_results,
    first_label="REINFORCE",
    second_label="VPG",
    first_color="orange",
    second_color="green",
)

The baseline makes a visible difference. VPG improves earlier and more steadily than REINFORCE, and the range across seeds is narrower: the same amount of experience produces more reliable progress, because the advantages carry less noise than the raw returns. This is pure **variance reduction**. The two algorithms estimate the same gradient in expectation, since any state-dependent baseline is unbiased, however inaccurate the value network may be.

Both remain unbiased estimators of the policy gradient, up to the two approximations we have accepted along the way: omitting the discount factor in front of later policy-gradient terms,

$\displaystyle \gamma^t$

and treating the Cart-Pole time limit as the end of the episode. Note, however, what has *not* changed: VPG still needs complete episodes, still uses every episode exactly once, and still shows fluctuations in the final phase of training, when a few unlucky episodes can push a good policy away from the solution. These remaining problems have a different origin, the Monte Carlo nature of the returns, and will be addressed in the next chapter.


## Solve Lunar Lander

Cart-Pole is a convenient test bed, but it is almost too easy to show the differences between algorithms. A more demanding classic problem is **Lunar Lander**: the agent controls a lander that must touch down softly on a landing pad, firing its main engine or one of the two lateral engines, or doing nothing.

The state has eight variables: six continuous variables, describing horizontal and vertical position and velocity, angle, and angular velocity, plus two binary flags for the legs touching the ground. The action space has four discrete actions. The reward is shaped: the agent gains points for moving towards the pad and for each leg that touches the ground, loses a small amount for every frame in which an engine fires, and receives a large penalty or bonus at the end:

$\displaystyle -100 \quad \text{for crashing}, \qquad +100 \quad \text{for coming to rest}$

The environment is considered solved when the average return exceeds

$\displaystyle 200$

The environment is part of the Box2D family of Gymnasium, which requires an additional installation (`pip install "gymnasium[box2d]"`):


In [ ]:
lunar_lander = gymnasium.make('LunarLander-v3', render_mode="rgb_array")

print('State Space: ', lunar_lander.observation_space.shape)
print('Action Space: ', lunar_lander.action_space.n)

Because our networks read the sizes of the spaces from the environment, nothing in the code needs to change: we create new networks and optimizers for the new environment and run the same algorithm. Lunar Lander episodes are longer, with a maximum length of

$\displaystyle 1000$

steps, and the task is harder, so we train for more episodes.

We also change the way we measure progress. On Lunar Lander the greedy policy can behave very differently from the stochastic policy it is derived from, especially early in training. The entropy bonus keeps the action probabilities spread out, and always taking the most probable action, even when it wins by a small margin, can drive the lander systematically off course. The greedy agent can therefore score much worse than the policy we are actually optimising.

We therefore evaluate the **stochastic policy**. Since the training episodes are generated by that very policy, their returns are already samples of its performance. We record them at no extra cost, instead of running separate evaluation episodes. The random policy is a useful reference also here, and it scores strongly negative, since it usually crashes:


In [ ]:
# set the seeds for reproducibility of results
seed = 34

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

lunar_lander.reset(seed=seed)
lunar_lander.action_space.seed(seed)

# fresh networks and optimizers for the new environment
pi = create_policy_network(lunar_lander, first_hidden_layer=128, second_hidden_layer=128)
v = create_value_network(lunar_lander, first_hidden_layer=128, second_hidden_layer=128)
policy_optimizer = torch.optim.RMSprop(pi.parameters(), lr=0.0005)
value_optimizer = torch.optim.RMSprop(v.parameters(), lr=0.0005)

random_score = evaluate(lunar_lander, lambda state: lunar_lander.action_space.sample(), episodes=100)
print("Random score: ", random_score)

In [ ]:
scores = vpg(lunar_lander, max_episodes=3000, eval_episodes=0)

We plot the learning curve of this single run, smoothed with a sliding window, and evaluate the final agent, both with the stochastic policy and with the greedy one:

In [ ]:
smoothing_window = 50
smoothed_scores = np.convolve(scores, np.ones(smoothing_window) / smoothing_window, mode="valid")

plt.figure(figsize=(12, 6))
plt.title("VPG on Lunar Lander (returns of the training episodes)")
plt.xlabel("Episodes")
plt.ylabel("Score")
plt.plot(np.arange(len(smoothed_scores)) + smoothing_window, smoothed_scores, color="green", linewidth=2)
plt.axhline(200, color="gray", linestyle="--", linewidth=1, label="Solved")
plt.legend()
plt.show()

In [ ]:
stochastic_score = evaluate(lunar_lander, stochastic_pi, episodes=10)
greedy_score = evaluate(lunar_lander, greedy_pi, episodes=10)

print("Random score", random_score)
print("Agent score (stochastic policy)", stochastic_score)
print("Agent score (greedy policy)", greedy_score)

In [ ]:
show_policy(lunar_lander, stochastic_pi, n_episodes=1, max_steps=1000)

The learning curve of Lunar Lander shows, on a larger scale, the behaviour we observed on Cart-Pole. In the first few hundred episodes the agent learns not to crash, and the score rises from the random-policy region,

$\displaystyle -200$

to roughly

$\displaystyle -100$

At this stage the lander often hovers, firing the engines. This costs fuel but avoids the crash penalty.

Learning to approach the pad and touch down is much slower: the curve climbs gradually, with large oscillations, and after

$\displaystyle 3000$

episodes a typical run has reached scores around zero, with occasional successful landings, still far from the solved threshold:

$\displaystyle 200$

The greedy policy, evaluated at the end, usually scores lower than the stochastic one, for the reason discussed above. This is typical of Monte Carlo policy gradient methods on tasks with long episodes: the returns are sums of up to

$\displaystyle 1000$

noisy rewards, so the advantages remain noisy even with a good baseline. Every episode produces a single gradient step, and many thousands of episodes would be needed to solve the task. The actor-critic methods of the next chapter will reduce this cost substantially.


## Limitations and Outlook

Policy gradient methods bring a new way of thinking about reinforcement learning: instead of learning *how much* each action is worth and deriving the behaviour from it, we learn *what to do* and adjust it in the direction that increases the expected return. The policy gradient theorem makes this possible without a model of the environment, and REINFORCE and VPG implement it with little more than a neural network and a loss function. The methods of this chapter also have clear limitations, and each of them points to one of the next chapters.

First, the gradient estimates have **high variance**, even with a baseline, because they are built from Monte Carlo returns. Replacing part of the return with a learned value estimate, that is **bootstrapping** as in temporal-difference learning, reduces the variance at the cost of some bias. This is the step from a baseline to a **critic**, and it is the subject of Chapter 09.

Second, the methods are **on-policy**: each trajectory is used for a single gradient step and then discarded, which makes them **sample inefficient** compared with methods that reuse experience from a replay memory. Chapter 10 will show how a deterministic policy can be combined with the off-policy machinery of DQN, and Chapter 11 will show how to take several gradient steps on the same batch of trajectories without breaking the on-policy assumption.

Third, the **step size is critical**. The policy parameters live in a space whose geometry has little to do with the space of action probabilities. A small change in the parameters,

$\displaystyle \theta$

can change the policy a lot in some states and very little in others:

$\displaystyle \pi_\theta(a \mid s)$

A step that is too large can collapse the policy irreversibly. Trust region methods, in Chapter 11, address exactly this problem.

Finally, we have only considered **discrete actions** and a softmax policy. The same theory applies to continuous actions, with a policy that outputs the parameters of a probability density, for example the mean and standard deviation of a Gaussian. The score function is computed in the same way. Continuous control is the subject of Chapter 10.
